In [1]:
from rdkit import Chem
from rdkit.Chem import AllChem

In [ ]:
def supplierfunc(file_path):
    supplier = Chem.SDMolSupplier(
            file_path,
            delimiter = " ",
            smilesColumn=0,
            nameColumn=1,
            titleLine=True,
            sanitize = True
            )
    for mol in supplier:
        if mol is not None:
            mol_obj = mol
            yield mol_obj

In [ ]:
def pocket_aware_ff_relaxation(ligand_supp_output, protein_dir, cutoff_distance = 6.0, max_iter = 100):
    ## Load the ligand and protein molecules
    ligand = next(supplierfunc(ligand_supp_output))
    protein = Chem.MolFromPDBFile(protein_dir)

    ## Generate 3D coordinates for the ligand if not present
    if ligand.GetNumConformers() == 0:
        AllChem.EmbedMolecule(ligand)

    pocket_editable = Chem.EditableMol(protein)
    atoms_to_keep = set()

    ## Identify atoms in the protein that are within the cutoff distance from the ligand
    for ligand_atom in ligand.GetAtoms():
        ligand_pos = ligand.GetConformer().GetAtomPosition(ligand_atom.GetIdx())
        for protein_atom in protein.GetAtoms():
            protein_pos = protein.GetConformer().GetAtomPosition(protein_atom.GetIdx())
            distance = ligand_pos.Distance(protein_pos)
            if distance <= cutoff_distance:
                atoms_to_keep.add(protein_atom.GetIdx())

    ## Remove all protein atoms that are not in the pocket
    ## purpose is to keep only the atoms in the cutoff to maximize compute efficiency
    for i in reversed(range(protein.GetNumAtoms())):
        if i not in atoms_to_keep:
            pocket_editable.RemoveAtom(i)

    pocket_mol = pocket_editable.GetMol()


    ## Combine the ligand and the pocket into a single molecule for force field purposes
    complex_mol = Chem.CombineMols(ligand, pocket_mol)
    num_protein_atoms = pocket_mol.GetNumAtoms()

    ## Create a force field for the combined molecule with the UFF force field because it takes into account
    # every atom type and is suitable for a wide range of molecules
    ff = AllChem.UFFGetMoleculeForceField(complex_mol)
    if ff is None:
        return ligand

    ## Freeze the pocket completely (force constant = effectively infinity)
    for i in range(num_protein_atoms):
        ff.UFFAddPositionConstraint(i, maxDistance=0.0, forceConstant=1e6)  # Using a very high force constant to effectively freeze the pocket

    ## Minimize the energy of the complex while keeping the pocket frozen
    ff.Minimize(maxIts=max_iter)

    ## Prepare to return the relaxed version of the original ligand back into molecule
    complex_conf = complex_mol.GetConformer()
    new_ligand_conf = Chem.Conformer(ligand.GetNumAtoms())

    for i in range(ligand.GetNumAtoms()):
        # append ligand atoms after protein atoms
        new_ligand_conf.SetAtomPosition(i, complex_conf.GetAtomPosition(num_protein_atoms + i))

    ligand.RemoveAllConformers()
    ligand.AddConformer(new_ligand_conf)

    yield ligand
    

    

In [4]:
def save_mol_to_file(mol, file_path):
    writer = Chem.SDWriter(file_path)
    writer.write(mol)
    print(f"Saved minimized ligand to {file_path}")

In [ ]:
file_dir = "C:\\augustine\\data\\output_diffdock_inferences\\test_inferences\\highest_confidence"
protein_dir = "C:\\augustine\\data\\main_protease\\cleaned_files\\4YLU_cleaned.pdb"
output_dir = "C:\\augustine\\data\\minimized_ligands"

In [ ]:
def main():
    supplier_output = supplierfunc(file_dir)
    save_mol_to_file(pocket_aware_ff_relaxation(supplier_output, protein_dir, cutoff_distance=6.0, max_iter=100), 
